# ShelfSense: FCOS dense shelf detector, one training run

The kernel only orchestrates. All training, checkpointing, `pycocotools` evaluation and ONNX export
lives in `training/train.py` in the repository, which this notebook clones at the start.
Checkpoints are written to `/kaggle/working/outputs/checkpoints` every epoch, before any export
or evaluation, so a failure later in the run never loses the trained weights.

The final cell prints exactly one `SUCCESS` or `FAILED` line and raises on failure, so Kaggle
reports ERROR instead of a misleading COMPLETE.

In [ ]:
import os, pathlib, shutil, subprocess, sys
os.environ["PYTHONUTF8"] = "1"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript", "pycocotools"], check=True)

REPO = pathlib.Path("/tmp/shelfsense_repo")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ibrahimSumon47/ShelfSense-Counter.git", str(REPO)], check=True)
print("cloned commit:", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

# Show what the datasets look like inside the kernel so a wrong mount is visible in the log.
for line in subprocess.run(["bash", "-lc", "ls /kaggle/input && find /kaggle/input -maxdepth 3 -type d | head -20"], capture_output=True, text=True).stdout.splitlines():
    print(line)

In [ ]:
OUT = pathlib.Path("/kaggle/working/outputs")
LOG = pathlib.Path("/kaggle/working/train.log")
OUT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(REPO / "training/train.py"),
    "--config", str(REPO / "training/configs/fcos-dense-v3.json"),
    "--cc0-dir", "/kaggle/input",
    "--sku-dir", "/kaggle/input",
    "--output-dir", str(OUT),
    "--eval-test-once",
]
print("$ " + " ".join(cmd), flush=True)

with LOG.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        log.write(line)
        log.flush()
        print(line.rstrip())
    returncode = proc.wait()

print("train.py exit code:", returncode)

In [ ]:
# Verify the run really produced what it claims, then say SUCCESS or FAILED in one line.
problems = []
if returncode != 0:
    problems.append(f"train.py exited with {returncode}")
if not (OUT / "metrics.json").is_file():
    problems.append("metrics.json missing")
if not list((OUT / "checkpoints").glob("*.pt")):
    problems.append("no checkpoint written")
if not list(OUT.glob("*.onnx")):
    problems.append("no ONNX export written")
for err in sorted(OUT.glob("*-error.txt")):
    problems.append(f"{err.name} exists")

print("output listing:")
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(f"  {path.stat().st_size / 1048576:8.1f} MB  {path}")

if (OUT / "metrics.json").is_file():
    import json
    report = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
    print("run_id:", report.get("run_id"))
    print("timings:", json.dumps(report.get("timings")))
    print("thresholds:", json.dumps(report.get("thresholds")))
    for key in ("val_sku110k", "val_cc0", "test_sku110k_once", "test_cc0_once"):
        block = report.get(key)
        if block:
            print(key, "map50", block.get("map50"), "mae", block.get("count_mae"), "mape", block.get("count_mape_pct"))

if problems:
    print("FAILED: " + "; ".join(problems))
    raise RuntimeError("ShelfSense training run failed: " + "; ".join(problems))
print("SUCCESS")